In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [3]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [6]:
X = df.drop("Class", axis = 1)
y = df["Class"]

In [11]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ANN

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [15]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [16]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [17]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [21]:
# creating the model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            
            nn.Linear(64,64),
            nn.ReLU(),
            
            nn.Linear(64,7) 
        )
        
    def forward(self, x):
        return self.model(x)

In [25]:
model = ANN()

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

epochs = 100

for epoch in range(epochs):
    model.train()
    
    running_loss = 0.0
    
    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    train_loss = running_loss/ len(train_loader)
    
    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.7263292851655379
epoch = 2/100, loss = 1.0894168174785117
epoch = 3/100, loss = 0.7013728514961575
epoch = 4/100, loss = 0.5202797158904697
epoch = 5/100, loss = 0.4292296767234802
epoch = 6/100, loss = 0.367710556672967
epoch = 7/100, loss = 0.3278182654277138
epoch = 8/100, loss = 0.2912906628587972
epoch = 9/100, loss = 0.27248871196871216
epoch = 10/100, loss = 0.2533198776452438
epoch = 11/100, loss = 0.23549900625063025
epoch = 12/100, loss = 0.22408079129198324
epoch = 13/100, loss = 0.20170190282489942
epoch = 14/100, loss = 0.20100074747334357
epoch = 15/100, loss = 0.19083956402281058
epoch = 16/100, loss = 0.1770387862039649
epoch = 17/100, loss = 0.17855019592072652
epoch = 18/100, loss = 0.16516673937439919
epoch = 19/100, loss = 0.15956683780836023
epoch = 20/100, loss = 0.1539144298952559
epoch = 21/100, loss = 0.14852453298542811
epoch = 22/100, loss = 0.14298216869001804
epoch = 23/100, loss = 0.14176040628682013
epoch = 24/100, loss = 0.1340545

In [28]:
#Evaluate
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1)
        
        correct += (predicted == yb).sum().item()
        total += yb.size(0)
        
print("correct vals:", total)
print("correct vals:", correct)

print("accuracy:", correct/total * 100)

correct vals: 180
correct vals: 172
accuracy: 95.55555555555556
